# 02 — SMLP4Rec + query embedding, and + sameDest (main model)

Query embedding: the current search (14 query fields, vocabulary from the train window only) enters SMLP4Rec as one extra position. Training rows: every train-window booking (warm and cold). Early stopping on **autoregressive valid NDCG@10**; dropout grid on the reference seed, then every seed.

Main model = query embedding + sameDest, a post-hoc fusion `log q_model + w · log q_sameDest` (smoothing alpha). alpha and w are chosen on the **autoregressive** valid rollout of the reference seed's model (one-SE rule by user; ties to the smaller w, then the larger alpha) and frozen for every seed. Both models are scored on valid and test under both protocols; the reference seed's AR test top-10 of the main model is saved for the CSV export.

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # torch before pandas-importing modules (Windows c10.dll)
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from nbp.data.split import TRAIN
from nbp.eval import runner as R
from nbp.eval.report import metrics, row_set_masks, write_json
from nbp.paths import CONFIGS, EXPERIMENTS, INTERIM, PROCESSED

SPLIT = yaml.safe_load((CONFIGS / "split.yaml").read_text(encoding="utf-8"))
TRAIN_CFG = yaml.safe_load((CONFIGS / "train.yaml").read_text(encoding="utf-8"))
MODEL_CFG = yaml.safe_load((CONFIGS / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
assert MODEL_CFG["MAX_ITEM_LIST_LENGTH"] == SPLIT["max_history"], "history length differs between configs"
SEEDS = list(SPLIT["seeds"])
REF_SEED = SEEDS[0]  # grids and sameDest weights are chosen on this seed
FAST = os.environ.get("NBP_FAST") == "1"  # smoke run: a user subsample, short training, outputs under *_fast
FAST_FRAC = float(os.environ.get("NBP_FAST_FRAC", "0.02"))
if FAST:
    TRAIN_CFG["max_epochs"] = int(os.environ.get("NBP_FAST_EPOCHS", "2"))
RES = ROOT / "results" / ("week6_retrain_fast" if FAST else "week6_retrain")
RANKS = INTERIM / ("week6_fast" if FAST else "week6")
EXP = EXPERIMENTS / "_fast" if FAST else EXPERIMENTS
TODAY = f"{date.today():%Y-%m-%d}"
K_MON = int(TRAIN_CFG["monitor"].split("@")[1])
BATCH = int(TRAIN_CFG["eval_batch"])
torch.set_num_threads(int(os.environ.get("NBP_THREADS", torch.get_num_threads())))

rows_full = R.load_rows(PROCESSED / "expedia_date_split" / "rows.npz")
codes = rows_full["codes"]  # (N, F) query codes, row = global event index
N_ITEMS = int(rows_full["y"].max()) + 1  # model ids 0 (padding) .. number of clusters
rows = R.subsample_users(rows_full, FAST_FRAC, 0) if FAST else rows_full
views = R.window_views(rows)
idx = R.window_index(rows)
Y = {w: rows["y"][i].astype(np.int64) for w, i in idx.items()}
L = {w: rows["L"][i] for w, i in idx.items()}
H = {w: rows["h"][i] for w, i in idx.items()}
USERS = {w: rows["user"][i] for w, i in idx.items()}
probe, y_probe = R.train_probe_view(rows, TRAIN_CFG["train_probe_rows"], TRAIN_CFG["train_probe_seed"])
print("FAST" if FAST else "FULL", {w: len(v) for w, v in views.items()}, "| items", N_ITEMS, "| threads", torch.get_num_threads())

FULL {'valid': 393399, 'test': 378788} | items 101 | threads 12


In [2]:
def summary_tables(ranks: dict) -> dict:
    """Metrics per window x protocol x row set (unrounded)."""
    out = {}
    for w in ("valid", "test"):
        masks = row_set_masks(L[w])
        for proto in ("ar", "tf"):
            out[f"{w}_{proto}"] = {rs: metrics(ranks[f"{w}_{proto}"], m) for rs, m in masks.items()}
    return out


def save_ranks(name: str, ranks: dict) -> Path:
    RANKS.mkdir(parents=True, exist_ok=True)
    path = RANKS / f"ranks_{name}.npz"
    np.savez_compressed(path, **{k: v for k, v in ranks.items()}, **{f"event_{w}": rows["event"][i] for w, i in idx.items()})
    return path

In [3]:
from nbp.hybrid.fusion import one_se_pick
from nbp.models.train import TrainConfig, fit, git_hash, run_dir_for

if FAST and TRAIN_CFG["min_delta"] is None:  # smoke run: use the smoke calibration of notebook 00
    TRAIN_CFG["min_delta"] = json.loads((RES / "min_delta_calibration.json").read_text(encoding="utf-8"))["min_delta"]
assert TRAIN_CFG["min_delta"] is not None, "run notebook 00 first (min_delta calibration)"


def train_cfg(seed: int) -> TrainConfig:
    return TrainConfig.from_dict(TRAIN_CFG, seed)


def per_row(r, k=K_MON):
    return (r <= k) / np.log2(r + 1)


def pick_dropout(valid_ranks: dict) -> tuple:
    """One-standard-error rule (user-clustered) on AR valid NDCG@k; ties go to the stronger
    regularisation (higher dropout)."""
    scores = {d: per_row(r) for d, r in valid_ranks.items()}
    return one_se_pick(scores, USERS["valid"], complexity=lambda d: -d)


def run_meta(run_dir: Path, res, extra: dict) -> dict:
    meta = {"run_dir": str(run_dir), "git_commit": git_hash(ROOT), "best_epoch": res.best_epoch,
            "best_monitor": res.best_monitor, "stop_reason": res.stop_reason, "per_epoch": res.per_epoch,
            "train_config": TRAIN_CFG, "model_config": MODEL_CFG, "split": SPLIT, **extra}
    write_json(run_dir / "metrics.json", meta)
    return meta

In [4]:
from nbp.data.query_features import build_vocab, encode
from nbp.models import scorers as S

train_rows = np.flatnonzero(rows["window"] == TRAIN)
vocab = build_vocab(codes, rows["event"][train_rows])  # train window only
Q = torch.from_numpy(encode(codes, vocab))
field_sizes = [len(v) + 1 for v in vocab]
oov = {w: [float((Q[rows["event"][i], j] == 0).float().mean()) for j in range(Q.shape[1])] for w, i in idx.items()}
loss = S.query_loss(rows["seq"][train_rows], rows["event"][train_rows], rows["y"][train_rows])
ev = R.valid_eval_fn(S.query_scorer, views["valid"], Y["valid"], probe, y_probe, K_MON, BATCH)
print("training rows (train window, all):", len(train_rows), "| vocab sizes", [len(v) for v in vocab])


def train_one(seed: int, drop: float):
    cfg = S.model_config(MODEL_CFG, {"seed": seed, "hidden_dropout_prob": drop})
    model = S.build_query(cfg, N_ITEMS, Q, field_sizes)
    run_dir = run_dir_for(EXP, "expedia", "query-embedding", f"d{drop}-s{seed}", TODAY)
    res = fit(model, len(train_rows), loss, ev, train_cfg(seed), run_dir)
    return model, res, run_dir

training rows (train window, all): 2180488 | vocab sizes [14711, 9, 12, 11, 14, 10, 10, 9, 2, 2, 11, 39, 5, 216]


## Dropout grid on the reference seed

In [5]:
grid = {}
for drop in TRAIN_CFG["dropout_grid"]:
    model, res, run_dir = train_one(REF_SEED, drop)
    ranks = R.evaluate(S.query_scorer(model), {"valid": views["valid"]}, Y, BATCH)
    grid[drop] = {"model": model, "res": res, "run_dir": run_dir, "valid_ar": ranks["valid_ar"]}
    print(drop, "best epoch", res.best_epoch, res.stop_reason, "| AR valid ndcg@10", round(per_row(ranks["valid_ar"]).mean(), 4))
DROP, best_d, eligible = pick_dropout({d: g["valid_ar"] for d, g in grid.items()})
print("chosen dropout", DROP, "| best", best_d, "| within one SE", eligible)

epoch 1: train loss 3.0649 (211.3s) | valid monitor 0.4319 nll 3.0217 (24.6s) | best 1, no gain 0/2


epoch 2: train loss 2.7712 (208.7s) | valid monitor 0.4399 nll 2.9431 (24.6s) | best 2, no gain 0/2


epoch 3: train loss 2.7150 (211.1s) | valid monitor 0.4407 nll 2.9515 (24.7s) | best 3, no gain 0/2


epoch 4: train loss 2.6853 (209.5s) | valid monitor 0.4436 nll 2.9441 (25.4s) | best 4, no gain 0/2


epoch 5: train loss 2.6646 (211.7s) | valid monitor 0.4441 nll 2.9309 (24.6s) | best 4, no gain 1/2


...

epoch 7: train loss 2.7729 (1057.7s) | valid monitor 0.4476 nll 2.9351 (24.7s) | best 6, no gain 1/2


epoch 8: train loss 2.7660 (1054.3s) | valid monitor 0.4481 nll 2.9230 (24.4s) | best 8, no gain 0/2


epoch 9: train loss 2.7602 (1052.5s) | valid monitor 0.4479 nll 2.9325 (24.4s) | best 8, no gain 1/2


epoch 10: train loss 2.7546 (1052.6s) | valid monitor 0.4495 nll 2.9187 (25.0s) | best 10, no gain 0/2
WARNING best epoch = max_epochs = 10: possibly under-trained
stop: max_epochs; best epoch 10 (monitor 0.4495)


0.2 best epoch 10 max_epochs | AR valid ndcg@10 0.4495
chosen dropout 0.2 | best 0.2 | within one SE [0.2]


## sameDest: alpha and w on the autoregressive valid rollout of the reference seed

In [6]:
SD = TRAIN_CFG["samedest"]
ref_model = grid[DROP]["model"]
base_ref = S.query_scorer(ref_model)
sd_valid = {}
t0 = time.time()
for a in SD["alpha_grid"]:
    for w in SD["w_grid"]:
        if w == 0.0 and sd_valid:
            # w = 0 is the query-embedding ranking for every alpha (smoothing is monotone): computed once
            sd_valid[(a, w)] = sd_valid[(SD["alpha_grid"][0], 0.0)]
            continue
        sc = R.score_view(views["valid"], S.samedest_scorer(base_ref, a, w, SD["recency"]), autoregressive=True, batch=BATCH)
        sd_valid[(a, w)] = R.target_rank(sc.scores, Y["valid"])
print(f"{len(sd_valid)} configurations in {time.time() - t0:.0f}s")
(ALPHA, W), best_cfg, eligible_cfg = one_se_pick({c: per_row(r) for c, r in sd_valid.items()}, USERS["valid"], complexity=lambda c: (c[1], -c[0]))
print("chosen alpha, w =", (ALPHA, W), "| best", best_cfg, "| within one SE:", len(eligible_cfg), "of", len(sd_valid))
sd_table = {f"alpha={a}, w={w}": float(per_row(r).mean()) for (a, w), r in sd_valid.items()}

65 configurations in 960s


chosen alpha, w = (0.1, 0.5) | best (0.05, 0.75) | within one SE: 9 of 65


## All seeds: query embedding and main model on valid and test, both protocols

In [7]:
from nbp.eval.metrics import target_rank

results = {}
for seed in SEEDS:
    if seed == REF_SEED:
        model, res, run_dir = grid[DROP]["model"], grid[DROP]["res"], grid[DROP]["run_dir"]
    else:
        model, res, run_dir = train_one(seed, DROP)
    q_sf = S.query_scorer(model)
    main_sf = S.samedest_scorer(q_sf, ALPHA, W, SD["recency"])
    rq = R.evaluate(q_sf, views, Y, BATCH, keep_top=10)
    rm = R.evaluate(main_sf, views, Y, BATCH, keep_top=10)
    save_ranks(f"query_s{seed}", rq)
    save_ranks(f"main_s{seed}", rm)
    common = {"seed": seed, "hidden_dropout_prob": DROP, "training_rows": "train window, warm + cold",
              "n_training_rows": int(len(train_rows)), "query_vocab_sizes": [len(v) for v in vocab], "query_oov_share": oov}
    meta_q = run_meta(run_dir, res, {"model": "SMLP4Rec + query embedding", **common, "results": summary_tables(rq)})
    write_json(RES / f"query_s{seed}.json", {**meta_q, "dropout_grid": {str(d): {"best_epoch": g["res"].best_epoch, "stop_reason": g["res"].stop_reason, "valid_ar_ndcg@10": float(per_row(g["valid_ar"]).mean())} for d, g in grid.items()}, "dropout_chosen": DROP})
    write_json(RES / f"main_s{seed}.json", {"model": "SMLP4Rec + query embedding + sameDest (main)", **common,
               "base_run_dir": str(run_dir), "samedest": {"alpha": ALPHA, "w": W, "recency": SD["recency"], "chosen_on": f"AR valid ndcg@{K_MON}, seed {REF_SEED}, one-SE rule by user", "best_on_valid": list(best_cfg), "n_within_one_se": len(eligible_cfg), "valid_grid_ndcg@10": sd_table},
               "results": summary_tables(rm)})
    results[seed] = (rq, rm)
    for nm, rr in (("query", rq), ("main", rm)):
        t = summary_tables(rr)["test_ar"]
        print(f"seed {seed} {nm}: test AR all ndcg@10 {t['all']['ndcg@10']:.4f} warm {t['warm']['ndcg@10']:.4f} cold {t['cold']['ndcg@10']:.4f}")

seed 2022 query: test AR all ndcg@10 0.4446 warm 0.4511 cold 0.4171
seed 2022 main: test AR all ndcg@10 0.4519 warm 0.4600 cold 0.4171


epoch 1: train loss 3.2239 (1041.7s) | valid monitor 0.4305 nll 3.0295 (25.0s) | best 1, no gain 0/2


epoch 2: train loss 2.9166 (1032.8s) | valid monitor 0.4406 nll 2.9816 (24.3s) | best 2, no gain 0/2


epoch 3: train loss 2.8650 (1041.7s) | valid monitor 0.4428 nll 2.9635 (24.3s) | best 3, no gain 0/2


epoch 4: train loss 2.8398 (1045.4s) | valid monitor 0.4444 nll 2.9448 (24.7s) | best 4, no gain 0/2


...

epoch 4: train loss 2.8145 (1070.5s) | valid monitor 0.4436 nll 2.9495 (25.0s) | best 4, no gain 0/2


epoch 5: train loss 2.7984 (1070.5s) | valid monitor 0.4463 nll 2.9393 (24.4s) | best 5, no gain 0/2


epoch 6: train loss 2.7868 (1070.9s) | valid monitor 0.4463 nll 2.9322 (25.0s) | best 5, no gain 1/2


epoch 7: train loss 2.7784 (1067.2s) | valid monitor 0.4469 nll 2.9334 (24.4s) | best 5, no gain 2/2
stop: patience; best epoch 5 (monitor 0.4463)


seed 2024 query: test AR all ndcg@10 0.4403 warm 0.4465 cold 0.4138
seed 2024 main: test AR all ndcg@10 0.4478 warm 0.4557 cold 0.4138


## Diagnostic: does sameDest confirm its own predictions? (reference seed, test, AR)

For each test row, the appended items are the earlier in-window predictions of the same user. A row is *self-confirming* if its top-1 equals an appended item whose booking was at the searched destination. Compared for the main model and the query embedding alone.

In [8]:
from nbp.eval.metrics import ndcg_at_k

v = views["test"]
ev_pos = {int(e): i for i, e in enumerate(v.event)}


def self_confirm(rank_dict) -> dict:
    top1 = rank_dict["test_ar_top"][:, 0].astype(np.int64)
    flag = np.zeros(len(v), bool)
    for i in np.flatnonzero(v.h > 0):
        for s in range(int(v.seq_len[i])):
            e = int(v.h_event[i, s])
            j = ev_pos.get(e)
            if j is not None and v.h_dest[i, s] == v.dest[i] and top1[j] == top1[i]:
                flag[i] = True
                break
    r = rank_dict["test_ar"]
    return {"rows": int(len(r)), "self_confirming_rows": int(flag.sum()), "share": float(flag.mean()),
            f"ndcg@{K_MON}_self_confirming": ndcg_at_k(r[flag], K_MON) if flag.any() else None,
            f"ndcg@{K_MON}_other": ndcg_at_k(r[~flag], K_MON)}


rq, rm = results[REF_SEED]
diag = {"query embedding": self_confirm(rq), "main (query embedding + sameDest)": self_confirm(rm)}
write_json(RES / "samedest_diagnostic.json", {"seed": REF_SEED, "window": "test", "protocol": "ar", **diag,
           "main_minus_query_embedding": "paired bootstrap per protocol (AR and TF) in bootstrap_ci.csv, written by notebook 04"})
print(json.dumps(diag, indent=1))

{
 "query embedding": {
  "rows": 378788,
  "self_confirming_rows": 44931,
  "share": 0.11861780204230335,
  "ndcg@10_self_confirming": 0.44943694188074135,
  "ndcg@10_other": 0.4439746943166194
 },
 "main (query embedding + sameDest)": {
  "rows": 378788,
  "self_confirming_rows": 45819,
  "share": 0.12096212129212118,
  "ndcg@10_self_confirming": 0.46420030906810017,
  "ndcg@10_other": 0.45016226365862383
 }
}
